# Player Level Progression — Simple

Minimal view of the **full population** (no A/B split) over a **90-day install lookback**:
1. Average + percentile (P10/P50/P90) max level reached by days since install
2. Cumulative level-reach funnel

All cohorts included have matured at least `days_since_install_cap` days, and the days-since-install axis is capped at that same value, so every cohort is observed over an equal, comparable window.

In [1]:
# hide-output
# Import libraries and initialise the BigQuery connector
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html

bqc = BigQueryConnector()

## Get data

### Setting parameters

In [2]:
# Complete population, no A/B windowing — but cohorts must be comparable:
# every included cohort has had at least `days_since_install_cap` days to mature,
# and no cohort is looked at beyond that many days since install.
import datetime as dt

days_since_install_cap = 28  # parameter: max days since install considered in every chart
install_lookback_days = 90   # parameter: how many days of install cohorts to include

end_date = dt.datetime.today() - dt.timedelta(days=1)                          # latest activity date available
install_cutoff_date = end_date - dt.timedelta(days=days_since_install_cap)      # newest cohort allowed — must have matured `days_since_install_cap` days
start_date = install_cutoff_date - dt.timedelta(days=install_lookback_days)     # oldest cohort included

print(f"Install window: {start_date.strftime('%Y-%m-%d')} to {install_cutoff_date.strftime('%Y-%m-%d')}")
print(f"Activity data through: {end_date.strftime('%Y-%m-%d')}")
print(f"Days since install cap: {days_since_install_cap}")

Install window: 2026-04-08 to 2026-07-07
Activity data through: 2026-08-04
Days since install cap: 28


In [3]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = False

### Max level by day

In [4]:
# hide-output
# Estimate query cost for player level SQL before executing
query_location = './sql/playerlevel_last90.sql'
parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'install_cutoff_date': install_cutoff_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'),
    'exclude_networks': ['CPE'],
}

cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 35.44 GB when run.
Estimated query cost: $0.24


In [5]:
# hide-output
# Fetch player level data from BigQuery or load from local pickle cache
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playerlevel_last90.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playerlevel_last90.pkl')

In [6]:
# hide-output
# Preview raw player level data
data

,user_id,dt,install_dt,country_code,days_since_install,max_level,platform,acquisition_type,install_build_version
0,6490D2D5DEE27D96,2026-06-04,2026-04-07,CA,58,27,IOS,Non-Attributed,0.72.1
1,F79AD8A0B5F1749D,2026-04-15,2026-04-07,HK,8,10,IOS,UA,0.72.1
2,ADB1C2C3A6F0921C,2026-05-04,2026-04-07,AU,27,16,AND,Non-Attributed,0.72.1
3,C526A4B7919609F,2026-04-29,2026-04-07,ES,22,12,AND,Non-Attributed,0.72.1
4,EEFD3323B6300DC6,2026-05-17,2026-04-07,SE,40,37,IOS,Non-Attributed,0.72.1
...,...,...,...,...,...,...,...,...,...
457707,B1805AE54704069E,2026-07-22,2026-07-06,IT,16,13,IOS,Non-Attributed,0.78.0
457708,282391CDAE903209,2026-07-07,2026-07-06,US,1,8,IOS,Non-Attributed,0.78.0
457709,81DD007DF98D0629,2026-07-23,2026-07-06,US,17,10,IOS,Non-Attributed,0.78.0
457710,CAA62CF1C05E0D7A,2026-07-21,2026-07-06,DE,15,16,IOS,Non-Attributed,0.78.0


### Max level per hour

In [7]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
minute_refresh_data = False

In [8]:
# hide-output
# Estimate query cost for the session-level Day 0 minute SQL before executing
minutes_since_install_cap = 1440  # parameter: max minutes since install considered — 1440 = Day 0 (first 24h)

minute_query_location = './sql/playerlevel_day0_minute.sql'
minute_parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'install_cutoff_date': install_cutoff_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'),
    'minutes_cap': minutes_since_install_cap,
    'exclude_networks': ['CPE'],
}

cost_info = bqc.print_cost_estimate(query=minute_query_location, is_path=True, query_parameters=minute_parameters)

This query will process 10.28 GB when run.
Estimated query cost: $0.07


In [9]:
# hide-output
# Fetch session-level Day 0 minute data from BigQuery or load from local pickle cache
minute_data = pd.DataFrame()

if minute_refresh_data:
    minute_data = bqc.get(query=minute_query_location, is_path=True, query_parameters=minute_parameters)
    minute_data.to_pickle('./data/playerlevel_day0_minute.pkl')
else:
    # Load from local cache to avoid repeated query costs
    minute_data = pd.read_pickle('./data/playerlevel_day0_minute.pkl')

In [10]:
# hide-output
# Preview raw session-level Day 0 data
minute_data

,user_id,session_id,session_start_ts,session_end_ts,install_ts,minutes_since_install,max_level,platform,acquisition_type,install_build_version,country_code
0,50960560B93EA22C,a7826bd5-515d-4764-9b4d-949480abc313,2026-06-24 04:06:05.376332+00:00,2026-06-24 04:42:54.913458+00:00,2026-06-24 04:05:25.919433+00:00,37,5,AND,Non-Attributed,0.62.0,US
1,562B9C2667AC468F,3b5c8dde-fa82-43ed-8846-d4b4e9562cb1,2026-04-15 05:46:51.968028+00:00,2026-04-15 06:08:03.327635+00:00,2026-04-15 05:46:20.861925+00:00,21,3,AND,Non-Attributed,0.67.1,US
2,562B9C2667AC468F,81cc47de-7baf-4838-a56d-ca425eec3975,2026-04-15 17:26:09.851645+00:00,2026-04-15 17:44:48.983017+00:00,2026-04-15 05:46:20.861925+00:00,718,4,AND,Non-Attributed,0.67.1,US
3,929BE1523BB4FEF5,614e5773-4242-41b0-9143-3499c2c9d0de,2026-06-22 18:54:12.286631+00:00,2026-06-22 19:04:48.144014+00:00,2026-06-22 18:43:35.136055+00:00,21,2,AND,Non-Attributed,0.51.0,US
4,D907F7DF14311379,de35e27e-f902-4c41-b71c-90d554eb6e79,2026-05-29 17:39:40.327987+00:00,2026-05-29 18:05:55.221317+00:00,2026-05-29 17:38:39.369738+00:00,27,3,AND,Non-Attributed,0.54.1,SE
...,...,...,...,...,...,...,...,...,...,...,...
76520,FCEAB2530DE17F11,1608897c-7f8b-447b-a3d9-d3ec239cbad3,2026-07-05 01:17:50.645654+00:00,2026-07-05 01:26:52.392536+00:00,2026-07-04 17:09:19.253061+00:00,497,5,IOS,Non-Attributed,0.78.0,US
76521,FEE24C8D9C89A259,f8692a9a-ca69-4900-8b9d-97a0e266e633,2026-07-01 05:46:25.212844+00:00,2026-07-01 06:00:59.478655+00:00,2026-07-01 05:46:25.212844+00:00,14,3,IOS,Non-Attributed,0.78.0,TH
76522,FF69FD6F10E1687A,bc070d38-7c1a-46f2-853c-fc1d36aaee04,2026-07-02 11:13:38.753784+00:00,2026-07-02 11:41:14.583994+00:00,2026-07-02 11:13:38.753784+00:00,27,3,AND,Non-Attributed,0.78.0,NL
76523,FF69FD6F10E1687A,c51c960b-599d-459d-b6c1-600adbe7f735,2026-07-02 16:44:33.263953+00:00,2026-07-02 17:16:53.403842+00:00,2026-07-02 11:13:38.753784+00:00,363,5,AND,Non-Attributed,0.78.0,NL


## Process data

In [11]:
# hide-output
# Cap every cohort's observation window to days_since_install_cap so results are comparable
# (the SQL install_cutoff_date already guarantees each included cohort could reach this many days)
data = data[data['days_since_install'] <= days_since_install_cap]

data

,user_id,dt,install_dt,country_code,days_since_install,max_level,platform,acquisition_type,install_build_version
1,F79AD8A0B5F1749D,2026-04-15,2026-04-07,HK,8,10,IOS,UA,0.72.1
2,ADB1C2C3A6F0921C,2026-05-04,2026-04-07,AU,27,16,AND,Non-Attributed,0.72.1
3,C526A4B7919609F,2026-04-29,2026-04-07,ES,22,12,AND,Non-Attributed,0.72.1
10,F4D4786FCE0361CB,2026-04-20,2026-04-07,US,13,15,AND,Non-Attributed,0.72.1
13,8CC071828AF6476F,2026-04-10,2026-04-07,MX,3,13,IOS,Non-Attributed,0.72.1
...,...,...,...,...,...,...,...,...,...
457707,B1805AE54704069E,2026-07-22,2026-07-06,IT,16,13,IOS,Non-Attributed,0.78.0
457708,282391CDAE903209,2026-07-07,2026-07-06,US,1,8,IOS,Non-Attributed,0.78.0
457709,81DD007DF98D0629,2026-07-23,2026-07-06,US,17,10,IOS,Non-Attributed,0.78.0
457710,CAA62CF1C05E0D7A,2026-07-21,2026-07-06,DE,15,16,IOS,Non-Attributed,0.78.0


## Player level progression

In [12]:
# hide-output
# Weighted average max level reached by days since install (drop buckets with <50 users)
min_bucket_size = 50

level_by_day = data.groupby(['days_since_install', 'max_level']).agg(
    users=('user_id', 'nunique')
).reset_index()

day_totals = level_by_day.groupby('days_since_install').agg(
    total_users=('users', 'sum')
).reset_index()

level_by_day = level_by_day.merge(day_totals, on='days_since_install')
level_by_day = level_by_day[level_by_day['users'] >= min_bucket_size]

avg_level_by_day = level_by_day.groupby('days_since_install').apply(
    lambda x: pd.Series({
        'weighted_avg_max_level': (x['max_level'] * x['users']).sum() / x['users'].sum(),
        'cohort_users': x['users'].sum(),
    }),
    include_groups=False
).reset_index()

avg_level_by_day

,days_since_install,weighted_avg_max_level,cohort_users
0,0,3.217705,50789.0
1,1,5.048697,21993.0
2,2,6.214795,15857.0
3,3,7.143039,13360.0
4,4,7.915167,11835.0
5,5,8.579220,10799.0
6,6,9.173853,10158.0
7,7,9.667649,9842.0
8,8,10.260464,9103.0
9,9,10.808813,8442.0


### Average + percentile max level reached

In [13]:
# hide-output
# P10 / P50 / P90 max level reached by days since install, weighted by user counts
def weighted_quantiles(group, quantiles=[0.1, 0.5, 0.9], measure_col='max_level'):
    levels = group[measure_col].values
    weights = group['users'].values
    sorted_idx = np.argsort(levels)
    levels, weights = levels[sorted_idx], weights[sorted_idx]
    cum_weights = np.cumsum(weights)
    total = cum_weights[-1]
    result = {}
    for q in quantiles:
        idx = np.searchsorted(cum_weights, q * total)
        result[f'p{int(q * 100)}'] = levels[min(idx, len(levels) - 1)]
    return pd.Series(result)

level_dist = data.groupby(['days_since_install', 'max_level']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts = level_dist.groupby('days_since_install').apply(
    weighted_quantiles, include_groups=False
).reset_index()

# Total users considered at each day bucket, for hover display
day_totals_for_pcts = level_dist.groupby('days_since_install')['users'].sum().reset_index(name='total_users')
level_pcts = level_pcts.merge(day_totals_for_pcts, on='days_since_install')

level_pcts

,days_since_install,p10,p50,p90,total_users
0,0,1,3,5,50845
1,1,3,5,7,22113
2,2,3,6,10,16006
3,3,4,7,11,13500
4,4,4,8,12,12016
5,5,5,9,13,11010
6,6,5,10,14,10375
7,7,5,10,15,10058
8,8,5,11,16,9324
9,9,6,11,17,8710


In [14]:
# Chart 1 — average + P10/P50/P90 max level band chart by days since install
df = level_pcts.merge(
    avg_level_by_day[['days_since_install', 'weighted_avg_max_level']],
    on='days_since_install', how='left',
).sort_values('days_since_install')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['p10'],
    mode='lines', line=dict(width=0), showlegend=False,
    customdata=df[['total_users']],
    hovertemplate='P10: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['p90'],
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(59,130,246,0.15)',
    name='P10–P90',
    customdata=df[['total_users']],
    hovertemplate='P90: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['p50'],
    mode='lines+markers', line=dict(width=2.5, color='rgba(59,130,246,1)'),
    name='P50 (median)',
    customdata=df[['total_users']],
    hovertemplate='P50: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['weighted_avg_max_level'],
    mode='lines', line=dict(width=1.5, color='rgba(30,41,59,0.85)', dash='dash'),
    name='Mean',
    hovertemplate='Mean: %{y:.2f}<extra></extra>',
))

fig.update_layout(
    title='Player level: average + P10/P50/P90 by days since install',
    xaxis_title='Days since install',
    yaxis_title='Max level',
    width=1000, height=500,
)
fig.show()

#### Summary table

In [15]:
# Same numbers as the chart above, in table form — easier to read exact values or paste elsewhere
days_summary_table = df[['days_since_install', 'weighted_avg_max_level', 'p10', 'p50', 'p90', 'total_users']].rename(columns={
    'days_since_install': 'Days since install',
    'weighted_avg_max_level': 'Mean level',
    'p10': 'P10',
    'p50': 'P50',
    'p90': 'P90',
    'total_users': 'Users',
})
days_summary_table['Mean level'] = days_summary_table['Mean level'].round(1)
days_summary_table

,Days since install,Mean level,P10,P50,P90,Users
0,0,3.2,1,3,5,50845
1,1,5.0,3,5,7,22113
2,2,6.2,3,6,10,16006
3,3,7.1,4,7,11,13500
4,4,7.9,4,8,12,12016
5,5,8.6,5,9,13,11010
6,6,9.2,5,10,14,10375
7,7,9.7,5,10,15,10058
8,8,10.3,5,11,16,9324
9,9,10.8,6,11,17,8710


## Level funnel

In [16]:
# hide-output
# Cumulative % of users reaching each level, based on each user's overall max level in the window
funnel_level_cap = 30  # a handful of users reach much higher levels; capped for readability

user_max_level = data.groupby('user_id')['max_level'].max()
total_users = user_max_level.shape[0]

levels = list(range(1, funnel_level_cap + 1))
funnel = pd.DataFrame({'level': levels})
funnel['users_reached'] = funnel['level'].apply(lambda l: (user_max_level >= l).sum())
funnel['pct_reached'] = funnel['users_reached'] / total_users

funnel

,level,users_reached,pct_reached
0,1,51404,1.000000
1,2,46263,0.899988
2,3,40167,0.781398
3,4,29992,0.583457
4,5,23092,0.449226
5,6,17849,0.347230
6,7,13516,0.262937
7,8,11884,0.231188
8,9,10707,0.208291
9,10,9918,0.192942


In [17]:
# Chart 2 — % of users reaching each level (cumulative), vertical bar chart with user counts
fig = go.Figure(go.Bar(
    x=[f'L{l}' for l in funnel['level']],
    y=funnel['pct_reached'],
    text=funnel['users_reached'].apply(lambda u: f'{u:,}'),
    textposition='outside',
    marker_color='rgba(59,130,246,0.85)',
    hovertemplate='%{x}: %{y:.1%} reached<br>n=%{text} users<extra></extra>',
))

fig.update_layout(
    title='% of users reaching each level (cumulative), labeled with user count',
    xaxis_title='Level',
    yaxis_title='% of users reached',
    yaxis_tickformat='.0%',
    width=1200, height=550,
    uniformtext_minsize=8,
    uniformtext_mode='hide',
)
fig.show()

## Level progression by minutes since install (Day 0 only)

Uses session-level data — `fact_ssdt_user_ssid` (precise `session_start_ts` per session) joined with `fact_ssdt_user_ssid_level_progression` (max level reached per session) — to get true per-minute resolution instead of calendar-day buckets. Restricted to each user's **Day 0** (their first `minutes_since_install_cap` minutes after install) — the window where minute-level resolution actually adds insight over the daily charts above. Same install-window population and maturity guarantee as above (`start_date` → `install_cutoff_date`).

In [18]:
# hide-output
# SQL already restricts to Day 0 (minutes_since_install_cap); collapse multiple sessions
# in the same minute bucket down to each user's highest level reached in that bucket
minute_data = minute_data[minute_data['minutes_since_install'] <= minutes_since_install_cap]
minute_data = minute_data.groupby(['user_id', 'minutes_since_install'])['max_level'].max().reset_index()

minute_data

,user_id,minutes_since_install,max_level
0,1000634FDA7838D2,10,2
1,100186873D03CE09,14,3
2,100186873D03CE09,51,4
3,100186873D03CE09,58,5
4,10048987539D074A,241,3
...,...,...,...
76456,FFFC6EC3E9B50ACC,1341,4
76457,FFFD05D0A3EFE3B4,3,2
76458,FFFE079DACB6000A,33,5
76459,FFFEB27F8529987F,27,5


In [19]:
# hide-output
# Weighted average max level reached by minutes since install (drop buckets with <50 users)
minute_level_by_minute = minute_data.groupby(['minutes_since_install', 'max_level']).agg(
    users=('user_id', 'nunique')
).reset_index()

minute_level_by_minute = minute_level_by_minute[minute_level_by_minute['users'] >= min_bucket_size]

minute_avg_level_by_minute = minute_level_by_minute.groupby('minutes_since_install').apply(
    lambda x: pd.Series({
        'weighted_avg_max_level': (x['max_level'] * x['users']).sum() / x['users'].sum(),
        'cohort_users': x['users'].sum(),
    }),
    include_groups=False
).reset_index()

minute_avg_level_by_minute

,minutes_since_install,weighted_avg_max_level,cohort_users
0,2,2.000000,317.0
1,3,2.000000,627.0
2,4,2.000000,895.0
3,5,2.064485,1039.0
4,6,2.154459,1256.0
5,7,2.291829,1285.0
6,8,2.425870,1322.0
7,9,2.533147,1433.0
8,10,2.627226,1572.0
9,11,2.744201,1595.0


### Average + percentile max level reached

In [20]:
# hide-output
# P10 / P50 / P90 max level reached by minutes since install, weighted by user counts
# (reuses the weighted_quantiles function defined earlier in the notebook)
minute_level_dist = minute_data.groupby(['minutes_since_install', 'max_level']).agg(
    users=('user_id', 'count')
).reset_index()

minute_level_pcts = minute_level_dist.groupby('minutes_since_install').apply(
    weighted_quantiles, include_groups=False
).reset_index()

# Total users considered at each minute bucket, for hover display
minute_totals = minute_level_dist.groupby('minutes_since_install')['users'].sum().reset_index(name='total_users')
minute_level_pcts = minute_level_pcts.merge(minute_totals, on='minutes_since_install')

minute_level_pcts

,minutes_since_install,p10,p50,p90,total_users
0,1,2,2,2,27
1,2,2,2,2,317
2,3,2,2,2,628
3,4,2,2,2,912
4,5,2,2,2,1039
...,...,...,...,...,...
1435,1436,3,5,7,28
1436,1437,4,6,8,31
1437,1438,3,5,8,28
1438,1439,4,5,7,41


In [21]:
# Chart 3 — average + P10/P50/P90 max level band chart by minutes since install (Day 0)
df_minute = minute_level_pcts.merge(
    minute_avg_level_by_minute[['minutes_since_install', 'weighted_avg_max_level']],
    on='minutes_since_install', how='left',
)
df_minute = df_minute[df_minute.minutes_since_install <= 60].sort_values('minutes_since_install')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['p10'],
    mode='lines', line=dict(width=0), showlegend=False,
    customdata=df_minute[['total_users']],
    hovertemplate='P10: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['p90'],
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(239,68,68,0.15)',
    name='P10–P90',
    customdata=df_minute[['total_users']],
    hovertemplate='P90: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['p50'],
    mode='lines', line=dict(width=2.5, color='rgba(239,68,68,1)'),
    name='P50 (median)',
    customdata=df_minute[['total_users']],
    hovertemplate='P50: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['weighted_avg_max_level'],
    mode='lines', line=dict(width=1.5, color='rgba(30,41,59,0.85)', dash='dash'),
    name='Mean',
    hovertemplate='Mean: %{y:.2f}<br>Minutes since install: %{x}<extra></extra>',
))

fig.update_layout(
    title='Player level: average + P10/P50/P90 by minutes since install (Day 0)',
    xaxis_title='Minutes since install',
    yaxis_title='Max level',
    width=1000, height=500,
)
fig.show()

#### Summary table

In [22]:
# Same numbers as the chart above, in table form — easier to read exact values or paste elsewhere
minute_summary_table = df_minute[['minutes_since_install', 'weighted_avg_max_level', 'p10', 'p50', 'p90', 'total_users']].rename(columns={
    'minutes_since_install': 'Minutes since install',
    'weighted_avg_max_level': 'Mean level',
    'p10': 'P10',
    'p50': 'P50',
    'p90': 'P90',
    'total_users': 'Users',
})
minute_summary_table['Mean level'] = minute_summary_table['Mean level'].round(1)
minute_summary_table

,Minutes since install,Mean level,P10,P50,P90,Users
0,1,NaN,2,2,2,27
1,2,2.0,2,2,2,317
2,3,2.0,2,2,2,628
3,4,2.0,2,2,2,912
4,5,2.1,2,2,2,1039
5,6,2.2,2,2,3,1258
6,7,2.3,2,2,3,1286
7,8,2.4,2,2,3,1330
8,9,2.5,2,3,3,1446
9,10,2.6,2,3,3,1593


## Hourly distribution of sessions started

When during the day do sessions start? Two variants: **Day 0** (reuses the session-level data above, now with `session_start_ts` added) and **All sessions** in the full observation window (new query — every session for the same matured install cohort, not just each user's first day).

`session_start_ts` is stored in UTC and there's no timezone/local-time field anywhere in the schema (checked `fact_ssdt_user_ssid`, `dim_user_install_device`, `dim_user_install_geo`, `dim_user_install_session`, `dimchange_user_install_ua`) — so hour-of-day below is in UTC, faceted by platform and colored by the top-5 countries by session volume (rest bucketed as "Other") as the closest available proxy for regional patterns.

**Note:** requires re-running the Day 0 data-pull cell above with `minute_refresh_data = True` once — `sql/playerlevel_day0_minute.sql` was just updated to also select `session_start_ts`/`country_code`, and the cached pickle predates that.

### Get data — all sessions

In [23]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
sessions_hourly_refresh_data = False

sessions_hourly_query_location = './sql/playerlevel_sessions_hourly.sql'
sessions_hourly_parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'install_cutoff_date': install_cutoff_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'),
    'exclude_networks': ['CPE'],
}

cost_info = bqc.print_cost_estimate(query=sessions_hourly_query_location, is_path=True, query_parameters=sessions_hourly_parameters)

This query will process 3.72 GB when run.
Estimated query cost: $0.03


In [24]:
# hide-output
# Fetch all-sessions data from BigQuery or load from local pickle cache
sessions_hourly = pd.DataFrame()

if sessions_hourly_refresh_data:
    sessions_hourly = bqc.get(query=sessions_hourly_query_location, is_path=True, query_parameters=sessions_hourly_parameters)
    sessions_hourly.to_pickle('./data/playerlevel_sessions_hourly.pkl')
else:
    sessions_hourly = pd.read_pickle('./data/playerlevel_sessions_hourly.pkl')

In [25]:
# hide-output
# Preview raw all-sessions data
sessions_hourly

,user_id,session_id,session_start_ts,platform,acquisition_type,country_code
0,694BC4177F780D05,7e42dcbb-ecf5-4ad7-b886-d52c0ee20554,2026-08-02 22:19:41.963208+00:00,IOS,Non-Attributed,GH
1,7AB6CE86CC3EBB92,5ace2df1-1ed2-4508-bacf-9bac39ae6ce1,2026-08-02 09:31:09.557398+00:00,IOS,Non-Attributed,KZ
2,CF2BB3B9CB4A024B,9e9c482c-8e52-4b18-b8f7-c232fea6a50b,2026-08-02 14:43:48.291475+00:00,AND,Non-Attributed,UY
3,76C772C238A89B74,ff9eec37-688f-4f4c-be36-9fa80fb057b3,2026-08-02 13:10:24.944668+00:00,IOS,UA,DZ
4,A11B6AF3D6DFFC4E,dfa8a7d5-451a-4f29-8cd1-0311b021eb43,2026-08-02 19:48:28.451357+00:00,IOS,Non-Attributed,PK
...,...,...,...,...,...,...
2236642,39C4CA23836F156C,b9c14ff3-4bfa-4f89-b639-95a64b291341,2026-04-12 13:32:25.877455+00:00,IOS,Non-Attributed,ZA
2236643,FE9C99E72F5BAC29,2312e1de-d8a1-4f59-bcf9-64e636ad08c3,2026-04-12 11:00:34.244715+00:00,AND,Non-Attributed,ZA
2236644,BAA9710F0383E139,e33bba1e-bef0-4b40-8d2c-f24bdf3ef1de,2026-04-12 03:23:45.800702+00:00,AND,Non-Attributed,ZA
2236645,2885E69E1E56197E,da40da24-82be-4a6f-92d1-99314ca39e1b,2026-04-12 13:38:53.351150+00:00,IOS,Non-Attributed,ZA


In [26]:
# hide-output
# Day 0 session-level data already has session_start_ts (after the SQL fix) — reload the pickle
# fresh under its own name rather than reusing `minute_data`, since that variable gets collapsed
# (session_id/session_start_ts/country_code dropped) by the level-progression processing above.
session_hourly_day0 = pd.read_pickle('./data/playerlevel_day0_minute.pkl')
session_hourly_day0 = session_hourly_day0[session_hourly_day0['minutes_since_install'] <= minutes_since_install_cap]
session_hourly_day0

,user_id,session_id,session_start_ts,session_end_ts,install_ts,minutes_since_install,max_level,platform,acquisition_type,install_build_version,country_code
0,50960560B93EA22C,a7826bd5-515d-4764-9b4d-949480abc313,2026-06-24 04:06:05.376332+00:00,2026-06-24 04:42:54.913458+00:00,2026-06-24 04:05:25.919433+00:00,37,5,AND,Non-Attributed,0.62.0,US
1,562B9C2667AC468F,3b5c8dde-fa82-43ed-8846-d4b4e9562cb1,2026-04-15 05:46:51.968028+00:00,2026-04-15 06:08:03.327635+00:00,2026-04-15 05:46:20.861925+00:00,21,3,AND,Non-Attributed,0.67.1,US
2,562B9C2667AC468F,81cc47de-7baf-4838-a56d-ca425eec3975,2026-04-15 17:26:09.851645+00:00,2026-04-15 17:44:48.983017+00:00,2026-04-15 05:46:20.861925+00:00,718,4,AND,Non-Attributed,0.67.1,US
3,929BE1523BB4FEF5,614e5773-4242-41b0-9143-3499c2c9d0de,2026-06-22 18:54:12.286631+00:00,2026-06-22 19:04:48.144014+00:00,2026-06-22 18:43:35.136055+00:00,21,2,AND,Non-Attributed,0.51.0,US
4,D907F7DF14311379,de35e27e-f902-4c41-b71c-90d554eb6e79,2026-05-29 17:39:40.327987+00:00,2026-05-29 18:05:55.221317+00:00,2026-05-29 17:38:39.369738+00:00,27,3,AND,Non-Attributed,0.54.1,SE
...,...,...,...,...,...,...,...,...,...,...,...
76520,FCEAB2530DE17F11,1608897c-7f8b-447b-a3d9-d3ec239cbad3,2026-07-05 01:17:50.645654+00:00,2026-07-05 01:26:52.392536+00:00,2026-07-04 17:09:19.253061+00:00,497,5,IOS,Non-Attributed,0.78.0,US
76521,FEE24C8D9C89A259,f8692a9a-ca69-4900-8b9d-97a0e266e633,2026-07-01 05:46:25.212844+00:00,2026-07-01 06:00:59.478655+00:00,2026-07-01 05:46:25.212844+00:00,14,3,IOS,Non-Attributed,0.78.0,TH
76522,FF69FD6F10E1687A,bc070d38-7c1a-46f2-853c-fc1d36aaee04,2026-07-02 11:13:38.753784+00:00,2026-07-02 11:41:14.583994+00:00,2026-07-02 11:13:38.753784+00:00,27,3,AND,Non-Attributed,0.78.0,NL
76523,FF69FD6F10E1687A,c51c960b-599d-459d-b6c1-600adbe7f735,2026-07-02 16:44:33.263953+00:00,2026-07-02 17:16:53.403842+00:00,2026-07-02 11:13:38.753784+00:00,363,5,AND,Non-Attributed,0.78.0,NL


### Process data

In [27]:
# hide-output
# Fixed categorical colors (dataviz palette, slots 1-5: blue/orange/aqua/yellow/magenta) —
# assigned once from the larger all-sessions sample so a given country keeps the same color
# in both charts below. "Other" gets muted gray, not a categorical hue, since it's a residual
# bucket rather than a distinct entity.
CATEGORICAL_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']
OTHER_COLOR = '#898781'

top_countries = sessions_hourly['country_code'].value_counts().nlargest(5).index.tolist()
country_bucket_colors = dict(zip(top_countries, CATEGORICAL_COLORS))
country_bucket_colors['Other'] = OTHER_COLOR
country_bucket_order = top_countries + ['Other']

def bucket_countries(df, country_col='country_code'):
    return df[country_col].where(df[country_col].isin(top_countries), 'Other')

for df in (sessions_hourly, session_hourly_day0):
    df['country_bucket'] = bucket_countries(df)
    df['start_hour'] = df['session_start_ts'].dt.hour

### Day 0

In [28]:
# Chart 4 — hour-of-day distribution, faceted by platform, colored by country bucket.
# pct is normalized WITHIN each (platform, country_bucket) — each country's line sums to 100%
# across the 24 hours, so curves are shape-comparable regardless of a country's overall volume
# (a platform-total basis would make high-volume countries sit above low-volume ones even with
# an identical diurnal shape).
def hourly_distribution_chart(df, title, hour_col='start_hour', hour_label='Hour of day (UTC)'):
    agg = df.groupby(['platform', hour_col, 'country_bucket']).size().reset_index(name='sessions')
    group_totals = agg.groupby(['platform', 'country_bucket'])['sessions'].transform('sum')
    agg['pct'] = agg['sessions'] / group_totals

    fig = px.line(
        agg, x=hour_col, y='pct', color='country_bucket', facet_col='platform',
        category_orders={'country_bucket': country_bucket_order, hour_col: list(range(24))},
        color_discrete_map=country_bucket_colors,
        markers=True,
        title=title,
        labels={hour_col: hour_label, 'pct': "% of that country's sessions", 'country_bucket': 'Country'},
    )
    fig.update_yaxes(tickformat='.0%')
    fig.update_layout(width=1100, height=500)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.show()
    return agg

day0_hourly_agg = hourly_distribution_chart(session_hourly_day0, 'Sessions started by hour of day (UTC) — Day 0')

#### Summary table

In [29]:
day0_hourly_table = day0_hourly_agg.pivot_table(
    index='start_hour', columns=['platform', 'country_bucket'], values='pct', fill_value=0
)
day0_hourly_table

platform             AND                                                    \
country_bucket        BR        CA        DE        GB     Other        US   
start_hour                                                                   
0               0.077805  0.081994  0.012063  0.017413  0.022777  0.060595   
1               0.070823  0.064309  0.006349  0.019279  0.022987  0.067348   
2               0.057855  0.067524  0.008254  0.009950  0.024982  0.065705   
3               0.048379  0.041801  0.021587  0.012438  0.026976  0.055302   
4               0.028928  0.045016  0.028571  0.009950  0.030230  0.038328   
5               0.012968  0.017685  0.043175  0.013682  0.034114  0.028837   
6               0.008978  0.028939  0.045079  0.024254  0.031175  0.031028   
7               0.003990  0.020900  0.050794  0.031716  0.038102  0.019894   
8               0.006484  0.004823  0.050159  0.029851  0.042406  0.016244   
9               0.011471  0.009646  0.045079  0.042289  0.045870  0.011133   
10              0.019451  0.011254  0.050794  0.040423  0.044820  0.017704   
11              0.027431  0.027331  0.043175  0.039179  0.051013  0.023179   
12              0.032419  0.033762  0.057778  0.054726  0.049124  0.028290   
13              0.038903  0.038585  0.048889  0.046642  0.054477  0.032123   
14              0.038404  0.045016  0.043175  0.055348  0.055841  0.032670   
15              0.050873  0.033762  0.054603  0.065920  0.053952  0.039423   
16              0.049377  0.046624  0.055238  0.067786  0.056366  0.043439   
17              0.060349  0.043408  0.064762  0.069030  0.051538  0.042891   
18              0.058354  0.046624  0.059048  0.083333  0.054582  0.050557   
19              0.052868  0.059486  0.074921  0.070274  0.055526  0.054572   
20              0.058354  0.041801  0.057778  0.062811  0.050383  0.054755   
21              0.047880  0.046624  0.033651  0.057214  0.039677  0.057675   
22              0.070823  0.067524  0.027937  0.042289  0.034009  0.063150   
23              0.066833  0.075563  0.017143  0.034204  0.029075  0.065158   

platform             IOS                                                    
country_bucket        BR        CA        DE        GB     Other        US  
start_hour                                                                  
0               0.089051  0.055747  0.010529  0.025411  0.025162  0.063556  
1               0.073966  0.066070  0.008253  0.016229  0.024405  0.064979  
2               0.053528  0.068135  0.008822  0.009182  0.023875  0.063613  
3               0.042336  0.053682  0.008822  0.007474  0.025124  0.055078  
4               0.024818  0.044735  0.012521  0.008969  0.031027  0.046373  
5               0.010706  0.029594  0.031303  0.017510  0.033410  0.032603  
6               0.009246  0.029594  0.036426  0.032031  0.038783  0.024694  
7               0.007786  0.019270  0.039556  0.047192  0.041243  0.015477  
8               0.006813  0.008259  0.056631  0.064061  0.046540  0.012063  
9               0.013139  0.014453  0.056061  0.064916  0.047410  0.011038  
10              0.019465  0.013765  0.060899  0.055734  0.046956  0.012119  
11              0.024818  0.019959  0.060330  0.054239  0.050361  0.020142  
12              0.028224  0.023400  0.049516  0.053385  0.055242  0.024865  
13              0.040389  0.030282  0.056346  0.046551  0.055053  0.035277  
14              0.040389  0.029594  0.058338  0.048046  0.056188  0.038862  
15              0.044769  0.050929  0.056061  0.052530  0.052632  0.042959  
16              0.048175  0.050241  0.054354  0.051463  0.050210  0.045121  
17              0.056934  0.045423  0.060046  0.050395  0.051459  0.047454  
18              0.047689  0.050241  0.067729  0.054025  0.048999  0.050697  
19              0.051582  0.053682  0.066875  0.061072  0.047902  0.052461  
20              0.055474  0.050241  0.062322  0.061926  0.047410  0.059403  
21              0.061314  0.061941  0.038702 

### All sessions

In [30]:
all_hourly_agg = hourly_distribution_chart(sessions_hourly, 'Sessions started by hour of day (UTC) — All sessions')

#### Summary table

In [31]:
all_hourly_table = all_hourly_agg.pivot_table(
    index='start_hour', columns=['platform', 'country_bucket'], values='pct', fill_value=0
)
all_hourly_table

platform             AND                                                    \
country_bucket        BR        CA        DE        GB     Other        US   
start_hour                                                                   
0               0.063218  0.059953  0.006526  0.016565  0.022347  0.058888   
1               0.062619  0.063800  0.006140  0.012287  0.022830  0.061521   
2               0.049014  0.062518  0.009201  0.008557  0.022620  0.058691   
3               0.031437  0.052543  0.018209  0.007153  0.024979  0.051619   
4               0.018485  0.042747  0.032696  0.012133  0.030834  0.040421   
5               0.011737  0.033948  0.044460  0.022028  0.034756  0.030857   
6               0.009016  0.023938  0.050680  0.033218  0.038563  0.021823   
7               0.007528  0.016956  0.055772  0.040173  0.040063  0.016618   
8               0.009233  0.011399  0.053967  0.045329  0.042384  0.013469   
9               0.017886  0.014071  0.057819  0.044561  0.044498  0.013150   
10              0.027264  0.016885  0.056094  0.048664  0.049419  0.018040   
11              0.030656  0.025648  0.054016  0.047830  0.051292  0.026596   
12              0.040361  0.030992  0.054725  0.049563  0.052306  0.032875   
13              0.043681  0.037582  0.049391  0.051297  0.052468  0.037737   
14              0.047581  0.046274  0.051840  0.051406  0.052920  0.041045   
15              0.053241  0.045597  0.051212  0.057199  0.053351  0.043261   
16              0.057195  0.043994  0.054241  0.058076  0.053823  0.050549   
17              0.057830  0.047877  0.060719  0.060753  0.052182  0.051337   
18              0.053948  0.049088  0.060316  0.062969  0.052202  0.051586   
19              0.055889  0.053113  0.055305  0.066458  0.053299  0.053782   
20              0.062238  0.054004  0.047409  0.067664  0.049450  0.054632   
21              0.062329  0.050869  0.035242  0.061389  0.042353  0.056143   
22              0.063127  0.054930  0.021851  0.044978  0.033393  0.056284   
23              0.064487  0.061271  0.012166  0.029751  0.027669  0.059076   

platform             IOS                                                    
country_bucket        BR        CA        DE        GB     Other        US  
start_hour                                                                  
0               0.073275  0.060484  0.009077  0.019422  0.024766  0.061164  
1               0.066754  0.066246  0.005609  0.013835  0.024646  0.065372  
2               0.052688  0.065134  0.005496  0.009686  0.025820  0.063057  
3               0.034491  0.057566  0.010142  0.009505  0.027492  0.053567  
4               0.017892  0.051841  0.019877  0.010510  0.032066  0.043085  
5               0.011748  0.038584  0.032036  0.015664  0.036385  0.030436  
6               0.007796  0.023833  0.038405  0.027229  0.039559  0.021191  
7               0.006593  0.014223  0.043572  0.037467  0.041174  0.014676  
8               0.008048  0.008351  0.047187  0.041204  0.044060  0.010684  
9               0.014389  0.007476  0.049125  0.045605  0.046477  0.009956  
10              0.024970  0.014168  0.053579  0.047092  0.048713  0.013603  
11              0.030179  0.018909  0.057171  0.048951  0.051077  0.021248  
12              0.036269  0.022465  0.055800  0.049795  0.052757  0.028240  
13              0.042161  0.032622  0.052672  0.051654  0.053320  0.035697  
14              0.045970  0.040809  0.054655  0.055713  0.054570  0.041790  
15              0.052724  0.044000  0.055811  0.057712  0.053925  0.045139  
16              0.056892  0.046826  0.056298  0.058566  0.051499  0.048199  
17              0.055419  0.048759  0.061477  0.058637  0.050248  0.050840  
18              0.054844  0.051658  0.069738  0.061932  0.048632  0.052739  
19              0.053443  0.054959  0.070736  0.066906  0.047729  0.054970  
20              0.056694  0.055324  0.063188  0.071116  0.045576  0.055790  
21              0.061077  0.058898  0.044853 

## Hourly distribution — standardized to local time (top countries only)

Same data as above, but the UTC hour is shifted by each country's approximate standard-time UTC offset — a single representative offset per country (e.g. US uses Eastern, Brazil uses Brasília time), ignoring DST and the fact that large countries (US, CA, BR, AU, RU) span multiple real timezones. Good enough to see whether the diurnal pattern is actually driven by local daytime/nighttime rather than looking artificially spread out in UTC — not precise per-user local time (the schema has no such field, as noted above).

Only the 5 specific countries already picked get an offset; **"Other" is excluded** from these charts since it's a mix of ~200 countries with no single defined offset.

In [32]:
# hide-output
# Approximate standard-time UTC offset per country (single representative offset — ignores DST
# and intra-country timezone spread, e.g. US/CA/BR/AU/RU all span multiple zones, and India's
# real +5:30 gets truncated to the hour). Good enough for a rough "is the diurnal pattern
# actually driven by local day/night" check, not precise per-user local time.
country_utc_offset = {
    'US': -5, 'GB': 0, 'DE': 1, 'BR': -3, 'CA': -5, 'FR': 1, 'VN': 7, 'AU': 10, 'IN': 5.5,
    'IT': 1, 'PL': 1, 'ES': 1, 'MX': -6, 'NL': 1, 'SE': 1, 'JP': 9, 'KR': 9, 'CN': 8, 'RU': 3,
    'TR': 3, 'EG': 2, 'ZA': 2, 'AR': -3, 'CL': -4, 'CO': -5, 'PE': -5, 'PH': 8, 'ID': 7,
    'MY': 8, 'SG': 8, 'TH': 7, 'PT': 0, 'AE': 4, 'SA': 3, 'NG': 1, 'KE': 3, 'PK': 5, 'BD': 6,
    'NZ': 12, 'IE': 0, 'BE': 1, 'CH': 1, 'AT': 1, 'DK': 1, 'NO': 1, 'FI': 2, 'GR': 2, 'RO': 2,
    'HU': 1, 'CZ': 1, 'UA': 2, 'IL': 2,
}

missing = [c for c in top_countries if c not in country_utc_offset]
if missing:
    print(f'WARNING: no UTC offset defined for {missing} — add to country_utc_offset before trusting the local-time charts below')

for df in (sessions_hourly, session_hourly_day0):
    known = df['country_bucket'] != 'Other'
    df.loc[known, 'local_hour'] = (
        (df.loc[known, 'start_hour'] + df.loc[known, 'country_bucket'].map(country_utc_offset)) % 24
    ).astype(int)

### Day 0 (local time)

In [33]:
day0_local_agg = hourly_distribution_chart(
    session_hourly_day0[session_hourly_day0['country_bucket'] != 'Other'],
    'Sessions started by local hour (approx.) — Day 0',
    hour_col='local_hour', hour_label='Local hour (approx.)',
)

#### Summary table

In [34]:
day0_local_table = day0_local_agg.pivot_table(
    index='local_hour', columns=['platform', 'country_bucket'], values='pct', fill_value=0
)
day0_local_table

platform             AND                                               IOS  \
country_bucket        BR        CA        DE        GB        US        BR   
local_hour                                                                   
0.0             0.048379  0.017685  0.017143  0.017413  0.028837  0.042336   
1.0             0.028928  0.028939  0.012063  0.019279  0.031028  0.024818   
2.0             0.012968  0.020900  0.006349  0.009950  0.019894  0.010706   
3.0             0.008978  0.004823  0.008254  0.012438  0.016244  0.009246   
4.0             0.003990  0.009646  0.021587  0.009950  0.011133  0.007786   
5.0             0.006484  0.011254  0.028571  0.013682  0.017704  0.006813   
6.0             0.011471  0.027331  0.043175  0.024254  0.023179  0.013139   
7.0             0.019451  0.033762  0.045079  0.031716  0.028290  0.019465   
8.0             0.027431  0.038585  0.050794  0.029851  0.032123  0.024818   
9.0             0.032419  0.045016  0.050159  0.042289  0.032670  0.028224   
10.0            0.038903  0.033762  0.045079  0.040423  0.039423  0.040389   
11.0            0.038404  0.046624  0.050794  0.039179  0.043439  0.040389   
12.0            0.050873  0.043408  0.043175  0.054726  0.042891  0.044769   
13.0            0.049377  0.046624  0.057778  0.046642  0.050557  0.048175   
14.0            0.060349  0.059486  0.048889  0.055348  0.054572  0.056934   
15.0            0.058354  0.041801  0.043175  0.065920  0.054755  0.047689   
16.0            0.052868  0.046624  0.054603  0.067786  0.057675  0.051582   
17.0            0.058354  0.067524  0.055238  0.069030  0.063150  0.055474   
18.0            0.047880  0.075563  0.064762  0.083333  0.065158  0.061314   
19.0            0.070823  0.081994  0.059048  0.070274  0.060595  0.063260   
20.0            0.066833  0.064309  0.074921  0.062811  0.067348  0.086131   
21.0            0.077805  0.067524  0.057778  0.057214  0.065705  0.089051   
22.0            0.070823  0.041801  0.033651  0.042289  0.055302  0.073966   
23.0            0.057855  0.045016  0.027937  0.034204  0.038328  0.053528   

platform                                                
country_bucket        CA        DE        GB        US  
local_hour                                              
0.0             0.029594  0.014229  0.025411  0.032603  
1.0             0.029594  0.010529  0.016229  0.024694  
2.0             0.019270  0.008253  0.009182  0.015477  
3.0             0.008259  0.008822  0.007474  0.012063  
4.0             0.014453  0.008822  0.008969  0.011038  
5.0             0.013765  0.012521  0.017510  0.012119  
6.0             0.019959  0.031303  0.032031  0.020142  
7.0             0.023400  0.036426  0.047192  0.024865  
8.0             0.030282  0.039556  0.064061  0.035277  
9.0             0.029594  0.056631  0.064916  0.038862  
10.0            0.050929  0.056061  0.055734  0.042959  
11.0            0.050241  0.060899  0.054239  0.045121  
12.0            0.045423  0.060330  0.053385  0.047454  
13.0            0.050241  0.049516  0.046551  0.050697  
14.0            0.053682  0.056346  0.048046  0.052461  
15.0            0.050241  0.058338  0.052530  0.059403  
16.0            0.061941  0.056061  0.051463  0.057582  
17.0            0.065382  0.054354  0.050395  0.059118  
18.0            0.065382  0.060046  0.054025  0.064467  
19.0            0.055747  0.067729  0.061072  0.063556  
20.0            0.066070  0.066875  0.061926  0.064979  
21.0            0.068135  0.062322  0.055093  0.063613  
22.0            0.053682  0.038702  0.039505  0.055078  
23.0            0.044735  0.025327  0.023062  0.046373

### All sessions (local time)

In [35]:
all_local_agg = hourly_distribution_chart(
    sessions_hourly[sessions_hourly['country_bucket'] != 'Other'],
    'Sessions started by local hour (approx.) — All sessions',
    hour_col='local_hour', hour_label='Local hour (approx.)',
)

#### Summary table

In [36]:
all_local_table = all_local_agg.pivot_table(
    index='local_hour', columns=['platform', 'country_bucket'], values='pct', fill_value=0
)
all_local_table

platform             AND                                               IOS  \
country_bucket        BR        CA        DE        GB        US        BR   
local_hour                                                                   
0.0             0.031437  0.033948  0.012166  0.016565  0.030857  0.034491   
1.0             0.018485  0.023938  0.006526  0.012287  0.021823  0.017892   
2.0             0.011737  0.016956  0.006140  0.008557  0.016618  0.011748   
3.0             0.009016  0.011399  0.009201  0.007153  0.013469  0.007796   
4.0             0.007528  0.014071  0.018209  0.012133  0.013150  0.006593   
5.0             0.009233  0.016885  0.032696  0.022028  0.018040  0.008048   
6.0             0.017886  0.025648  0.044460  0.033218  0.026596  0.014389   
7.0             0.027264  0.030992  0.050680  0.040173  0.032875  0.024970   
8.0             0.030656  0.037582  0.055772  0.045329  0.037737  0.030179   
9.0             0.040361  0.046274  0.053967  0.044561  0.041045  0.036269   
10.0            0.043681  0.045597  0.057819  0.048664  0.043261  0.042161   
11.0            0.047581  0.043994  0.056094  0.047830  0.050549  0.045970   
12.0            0.053241  0.047877  0.054016  0.049563  0.051337  0.052724   
13.0            0.057195  0.049088  0.054725  0.051297  0.051586  0.056892   
14.0            0.057830  0.053113  0.049391  0.051406  0.053782  0.055419   
15.0            0.053948  0.054004  0.051840  0.057199  0.054632  0.054844   
16.0            0.055889  0.050869  0.051212  0.058076  0.056143  0.053443   
17.0            0.062238  0.054930  0.054241  0.060753  0.056284  0.056694   
18.0            0.062329  0.061271  0.060719  0.062969  0.059076  0.061077   
19.0            0.063127  0.059953  0.060316  0.066458  0.058888  0.063826   
20.0            0.064487  0.063800  0.055305  0.067664  0.061521  0.071856   
21.0            0.063218  0.062518  0.047409  0.061389  0.058691  0.073275   
22.0            0.062619  0.052543  0.035242  0.044978  0.051619  0.066754   
23.0            0.049014  0.042747  0.021851  0.029751  0.040421  0.052688   

platform                                                
country_bucket        CA        DE        GB        US  
local_hour                                              
0.0             0.038584  0.016182  0.019422  0.030436  
1.0             0.023833  0.009077  0.013835  0.021191  
2.0             0.014223  0.005609  0.009686  0.014676  
3.0             0.008351  0.005496  0.009505  0.010684  
4.0             0.007476  0.010142  0.010510  0.009956  
5.0             0.014168  0.019877  0.015664  0.013603  
6.0             0.018909  0.032036  0.027229  0.021248  
7.0             0.022465  0.038405  0.037467  0.028240  
8.0             0.032622  0.043572  0.041204  0.035697  
9.0             0.040809  0.047187  0.045605  0.041790  
10.0            0.044000  0.049125  0.047092  0.045139  
11.0            0.046826  0.053579  0.048951  0.048199  
12.0            0.048759  0.057171  0.049795  0.050840  
13.0            0.051658  0.055800  0.051654  0.052739  
14.0            0.054959  0.052672  0.055713  0.054970  
15.0            0.055324  0.054655  0.057712  0.055790  
16.0            0.058898  0.055811  0.058566  0.057378  
17.0            0.057585  0.056298  0.058637  0.059260  
18.0            0.059280  0.061477  0.061932  0.061920  
19.0            0.060484  0.069738  0.066906  0.061164  
20.0            0.066246  0.070736  0.071116  0.065372  
21.0            0.065134  0.063188  0.064394  0.063057  
22.0            0.057566  0.044853  0.047153  0.053567  
23.0            0.051841  0.027311  0.030253  0.043085

In [37]:
# hide-output
# Export notebook to HTML for sharing and archival
export_notebook_html(
    notebook_path='./level_progression_simple.ipynb',
    output_path='./level_progression_simple.html',
)

Saved to level_progression_simple.html


PosixPath('level_progression_simple.html')